In [2]:
# =============================================================================
# HOW TO RUN THIS FRESH (Colab / clean runtime)
# -----------------------------------------------------------------------------
# 0) Clean start
#    - Runtime > Restart runtime (recommended).
#
# 1) Set the switches near the top of this file:
#    NORM         = "none"      # for continuity with earlier results
#    USE_TTA      = False       # fairness: OFF for ablation reporting
#    RUN_SINGLE   = True        # run the ablation suite once
#    RUN_FINAL_OF_BEST = True   # retrain best ablation on Train∪Val for best-epoch; test across seeds
#    RUN_MULTI    = False       # (optional) multi-seed for every config; heavy
#    RUN_ULTIMATE = False       # (optional) combine winning knobs into one config
#    BATCH_SIZE   = 256         # keep fixed across runs
#    NUM_EPOCHS   = 100         # ablation/FINAL length
#
# 2) Execute order (top-to-bottom, once):
#    a) Imports + Drive mount + dataset load
#    b) Split + make_loaders_with_snr(normalize=NORM)
#    c) Define model/optimizer/scheduler + train/eval helpers
#    d) Run ablations: results = run_all(num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE)
#    e) If RUN_FINAL_OF_BEST=True: auto-pick winner, retrain on Train∪Val for best-epoch, test seeds [42,43,44]
#
# 3) Outputs (timestamped run folder):
#    BASE_DIR = /content/drive/MyDrive/amc_runs/amc1d_<YYYYMMDD-HHMMSS>/
#      ├─ models/   (ablation ckpts, FINAL_* for 3 seeds)
#      └─ plots/    (val/loss curves, SNR curves, AUC-over-SNR bar, BEST_* plots)
#    Also: per-config *_manifest.json with metrics and config.
#
# 4) Optional: TTA (test-time augmentation) *after* training (no retrain)
#    - Set USE_TTA = True
#    - Re-run only the evaluation/analysis cell(s) that:
#        • evaluate(model, test_loader, ...)
#        • evaluate_snr_curve(model, test_loader_snr)
#        • (optionally) seed-ensemble evaluation over FINAL checkpoints
#      This reports TTA uplift fairly, separate from ablations.
#
# 5) Optional: Multi-seed ablations for EVERY config
#    - Set RUN_MULTI=True and set NUM_MULTI_SEEDS
#    - Use only if you want mean±std per config (heavier compute).
#
# 6) Optional: ULTIMATE combined config
#    - Set RUN_ULTIMATE=True to build a config by win-vs-control voting.
#    - It trains and (optionally) retrains Train∪Val for best-epoch length.
#
# Notes:
#  - Keep USE_TTA=False for apples-to-apples ablation tables.
#  - Do not change batch size/epochs between ablation and FINAL unless you
#    re-run everything for fairness.
#  - All artifacts are versioned by timestamp to avoid clobbering prior runs.
# =============================================================================

# ========= Colab & Paths (mount first!) =========
import sys, os, platform, shutil
print(f"Python: {sys.version.split()[0]}  |  Platform: {platform.platform()}")

from google.colab import drive
drive.mount('/content/drive')
data_path = "/content/drive/MyDrive/AMC_datasets/RadioML2016_10A_dataset.pkl"

# ========= Determine what sections to run =========
RUN_SINGLE = True
RUN_FINAL_OF_BEST = True
RUN_ULTIMATE = False      # set True if you want the “combined” config run
RUN_MULTI = False         # set True to run the multi-seed ablations at the end
BATCH_SIZE = 256
# Prepocessing Data (this allows for Raw IQ)
NORM = "none"  # choose: "none" | "zscore" | "rms"
USE_TTA = False # <- set to False for fairness; True to show deploy-time uplift
TTA_SHIFTS = (0, 4, 8)  # tiny, fast


from datetime import datetime

# Top-level runs folder (same parent as traditional models)
BASE_ROOT = "/content/drive/MyDrive/amc_runs"

# Subfolder for this family of experiments (parallel to "traditional_models")
FAMILY = "cnn_1d"   # you can also use "cnn_models" if you prefer

# Timestamped run folder under that family
RUN_TAG   = datetime.now().strftime("%Y%m%d-%H%M%S")
DRIVE_ROOT = os.path.join(BASE_ROOT, FAMILY, RUN_TAG)

# Subfolders like in the other script
PLOTS_DIR  = os.path.join(DRIVE_ROOT, "plots")
MODELS_DIR = os.path.join(DRIVE_ROOT, "models")

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print("Artifacts will go to:", DRIVE_ROOT)


# ========= Clone repo & import loader =========
REPO_URL = "https://github.com/Koosmaster/Tutorial-Automatic-Modulation-Classification.git"
REPO_DIR = "/content/Tutorial-Automatic-Modulation-Classification"
SRC_DIR  = os.path.join(REPO_DIR, "src")
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 "$REPO_URL" "$REPO_DIR"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from data import load_radioml_pkl_dataset  # ← helper

# ========= Standard libs =========
import math, time, json, random, pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional
import subprocess

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import SGD, Adam
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR, SequentialLR, LinearLR
from torch.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ========= Repro seed =========
def set_global_seed(seed: int = 42) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ========= Load dataset from PKL =========
print("Loading RadioML 2016.10A from:", data_path)
radioml_data, mod_classes, snr_values = load_radioml_pkl_dataset(data_path)

# Convert dict -> arrays X (N,2,128), y (labels as strings), snr_list (N,)
X, y, snr_list = [], [], []
for (mod, snr), signals in radioml_data.items():
    for complex_signal in signals:
        iq = np.vstack((complex_signal.real, complex_signal.imag))  # (2, N)
        X.append(iq)
        y.append(mod)
        snr_list.append(snr)
X = np.array(X, dtype=np.float32)           # (N,2,128)
y = np.array(y)
snr_list = np.array(snr_list)
print("X shape:", X.shape, "| y shape:", y.shape, "| snr shape:", snr_list.shape)
print("Unique mods:", sorted(list(set(y))), "| Unique SNRs:", sorted(list(set(snr_list))))

# Encode labels once (keeps stable class order)
le = LabelEncoder()
y_enc = le.fit_transform(y).astype(np.int64)
label_names = list(le.classes_)
num_classes = len(label_names)

# Stratified 70/15/15 split
X_train, X_tmp, y_train, y_tmp, snr_train, snr_tmp = train_test_split(
    X, y_enc, snr_list, test_size=0.30, stratify=y_enc, random_state=42
)
X_val, X_test, y_val, y_test, snr_val, snr_test = train_test_split(
    X_tmp, y_tmp, snr_tmp, test_size=0.50, stratify=y_tmp, random_state=42
)


#============ TTA Trick for Low SNR ============
def _predict_with_tta(model, xb, shifts=(0,4,8)):
    # xb: (B,2,128)
    logits_sum = 0
    for s in shifts:
        if s == 0:
            x = xb
        else:
            x = torch.roll(xb, shifts=s, dims=-1)
        logits_sum = logits_sum + model(x)
    return logits_sum / len(shifts)

# ========= Normalization & Loaders =========
def _norm_none(x_np: np.ndarray) -> np.ndarray:
    return x_np

def _norm_zscore(x_np: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    m = x_np.mean(axis=2, keepdims=True)
    s = x_np.std(axis=2, keepdims=True) + eps
    return (x_np - m) / s

def _norm_rms(x_np: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    rms = np.sqrt((x_np * x_np).mean(axis=2, keepdims=True)) + eps
    return x_np / rms

def _get_norm_fn(mode: str):
    if mode == "none":   return _norm_none
    if mode == "zscore": return _norm_zscore     # your current behavior
    if mode == "rms":    return _norm_rms        # AGC-like scaling
    raise ValueError(f"Unknown normalize mode: {mode}")

def make_loaders_with_snr(
    X_tr, y_tr, snr_tr, X_va, y_va, snr_va, X_te, y_te, snr_te,
    batch_size=256, seed=42, normalize: str = "zscore"
):
    g = torch.Generator(device="cpu").manual_seed(seed)
    pin = (device.type == "cuda")
    norm_fn = _get_norm_fn(normalize)

    Xt = torch.from_numpy(norm_fn(X_tr)).float()
    Xv = torch.from_numpy(norm_fn(X_va)).float()
    Xe = torch.from_numpy(norm_fn(X_te)).float()
    yt = torch.from_numpy(y_tr).long()
    yv = torch.from_numpy(y_va).long()
    ye = torch.from_numpy(y_te).long()
    st = torch.from_numpy(snr_tr).float()
    sv = torch.from_numpy(snr_va).float()
    se = torch.from_numpy(snr_te).float()

    train_loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True,  generator=g, pin_memory=pin)
    val_loader   = DataLoader(TensorDataset(Xv, yv), batch_size=batch_size, shuffle=False, generator=g, pin_memory=pin)
    test_loader  = DataLoader(TensorDataset(Xe, ye), batch_size=batch_size, shuffle=False, generator=g, pin_memory=pin)
    test_loader_snr = DataLoader(TensorDataset(Xe, ye, se), batch_size=batch_size, shuffle=False, generator=g, pin_memory=pin)
    return train_loader, val_loader, test_loader, test_loader_snr


train_loader, val_loader, test_loader, test_loader_snr = make_loaders_with_snr(
    X_train, y_train, snr_train, X_val, y_val, snr_val, X_test, y_test, snr_test,
    batch_size=256, seed=42, normalize=NORM
)
NORMALIZE_MODE = NORM



# ========= Model =========
def conv_block(in_c, out_c, k=3, s=1):
    p = k // 2
    return nn.Sequential(
        nn.Conv1d(in_c, out_c, kernel_size=k, stride=s, padding=p, bias=False),
        nn.BatchNorm1d(out_c),
        nn.ReLU(inplace=True),
    )

class AMC1DCNN(nn.Module):
    # (B,2,128) → [Conv-BN-ReLU]*2 + MaxPool(2) ×3 → GAP → 128→256→Dropout→C
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(2, 32), conv_block(32, 32), nn.MaxPool1d(2),  # 128→64
            conv_block(32, 64), conv_block(64, 64), nn.MaxPool1d(2), # 64→32
            conv_block(64,128), conv_block(128,128), nn.MaxPool1d(2) # 32→16
        )
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(128,256), nn.ReLU(inplace=True), nn.Dropout(0.5), nn.Linear(256, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if getattr(m, "bias", None) is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)          # (B,128,16)
        x = self.gap(x).squeeze(-1)   # (B,128)
        return self.classifier(x)     # (B,C)

# ========= Extra helpers for plots =========
def best_epoch_from_history(hist):  # 1-based
    return int(np.argmax(np.array(hist["val_acc"])) + 1)


def compute_auc_over_snr(snrs, accs_percent, normalize=True):
    """accs_percent: list of accuracies in PERCENT; returns scalar (percent units)."""
    s = np.array(snrs, dtype=float)
    a = np.array(accs_percent, dtype=float)
    area = np.trapz(a, s)  # percent * dB
    if not normalize:
        return float(area)
    span = float(s.max() - s.min()) if s.size > 1 else 1.0
    return float(area / max(span, 1e-8))  # ≈ mean accuracy across SNR in percent

@torch.no_grad()
def eval_seed_ensemble(model_paths, test_loader, num_classes):
    models = []
    for p in model_paths:
        m = AMC1DCNN(num_classes).to(device)
        m.load_state_dict(torch.load(p, map_location=device))
        m.eval()
        models.append(m)

    correct, total = 0, 0
    for xb, yb in test_loader:
        xb = xb.to(device); yb = yb.to(device)
        logits = 0
        for m in models:
            if USE_TTA:
                logits = logits + _predict_with_tta(m, xb, shifts=TTA_SHIFTS)
            else:
                logits = logits + m(xb)
        preds = logits.argmax(1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    return 100.0 * correct / max(1, total)


@torch.no_grad()
def _collect_preds_targets(model, loader, want_probs=False):
    model.eval()
    preds, tgts, probs = [], [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        out = model(xb)
        if want_probs:
            p = torch.softmax(out, dim=1).cpu().numpy()
            probs.append(p)
        preds.append(out.argmax(1).cpu().numpy())
        tgts.append(yb.numpy())
    preds = np.concatenate(preds); tgts = np.concatenate(tgts)
    if want_probs:
        probs = np.concatenate(probs)
        return preds, tgts, probs
    return preds, tgts

@torch.no_grad()
def plot_confusion_at_snr(ckpt_path: str, name: str, target_snr: float, label_names, save_dir: str):
    """Confusion at an SNR (uses global `test_loader_snr`)."""
    model = AMC1DCNN(num_classes).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    xs, ys = [], []
    for xb, yb, sb in test_loader_snr:
        idx = np.isclose(sb.cpu().numpy(), target_snr)
        if not idx.any(): continue
        xs.append(xb[idx]); ys.append(yb[idx])
    if not xs:
        print(f"[{name}] No samples at SNR={target_snr} dB; skipping confusion matrix.")
        return
    Xs = torch.cat(xs, dim=0).to(device)
    Ys = torch.cat(ys, dim=0).cpu().numpy()
    with torch.no_grad():
        logits = model(Xs)
        Yh = logits.argmax(1).cpu().numpy()

    # Confusion matrix
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(Ys, Yh, labels=list(range(len(label_names))))

    # Plot
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(cm, interpolation="nearest", aspect="auto")
    ax.set_xticks(range(len(label_names))); ax.set_xticklabels(label_names, rotation=90)
    ax.set_yticks(range(len(label_names))); ax.set_yticklabels(label_names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{name} — Confusion @ {target_snr:.0f} dB (N={cm.sum()})")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fp = os.path.join(save_dir, f"{name}_confusion_{int(target_snr)}dB.png")
    plt.savefig(fp, dpi=300, bbox_inches="tight"); plt.close()
    print("Saved:", fp)

@torch.no_grad()
def plot_reliability_and_ece(ckpt_path: str, name: str, test_loader, save_dir: str, n_bins: int = 15):
    """Reliability diagram + ECE over the WHOLE test set."""
    model = AMC1DCNN(num_classes).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    preds, tgts, probs = _collect_preds_targets(model, test_loader, want_probs=True)
    conf = probs.max(axis=1)
    correct = (preds == tgts).astype(np.float32)

    # Bin stats
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(conf, bins) - 1
    bin_acc, bin_conf, bin_count = [], [], []
    ece = 0.0
    for b in range(n_bins):
        sel = bin_ids == b
        count = sel.sum()
        if count == 0:
            bin_acc.append(0.0); bin_conf.append(0.0); bin_count.append(0)
            continue
        acc_b = float(correct[sel].mean())
        conf_b = float(conf[sel].mean())
        bin_acc.append(acc_b); bin_conf.append(conf_b); bin_count.append(int(count))
        ece += (count / len(conf)) * abs(acc_b - conf_b)

    # Plot reliability
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0,1], [0,1])
    ax.bar((bins[:-1] + bins[1:]) / 2.0, bin_acc, width=(1.0/n_bins), align="center", alpha=0.7)
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.set_title(f"{name} — Reliability (ECE={ece*100:.2f}%)")
    fig.tight_layout()
    fp = os.path.join(save_dir, f"{name}_reliability.png")
    plt.savefig(fp, dpi=300, bbox_inches="tight"); plt.close()
    print("Saved:", fp)

def plot_lr_curves_for_all(results, save_dir: str):
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10,5))
    for r in results:
        lr = r["history"].get("lr", None)
        if lr is None or len(lr) == 0: continue
        plt.plot(lr, label=r["name"])
    plt.xlabel("Epoch"); plt.ylabel("LR")
    plt.title("Learning Rate vs Epoch (All Configs)")
    plt.legend()
    fp = os.path.join(save_dir, "lr_curves_all_configs.png")
    plt.savefig(fp, dpi=300, bbox_inches="tight"); plt.close()
    print("Saved:", fp)


# ========= Optimizer / Scheduler builders =========
def build_param_groups(model: nn.Module, no_bias_decay: bool):
    if not no_bias_decay:
        return [{"params": [p for p in model.parameters() if p.requires_grad], "weight_decay": 1e-4}]
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        if p.ndim == 1 or name.endswith(".bias") or "bn" in name.lower():
            no_decay.append(p)
        else:
            decay.append(p)
    return [{"params": decay, "weight_decay": 1e-4}, {"params": no_decay, "weight_decay": 0.0}]

def build_optimizer(model: nn.Module, cfg):
    pg = build_param_groups(model, no_bias_decay=cfg.no_bias_decay)
    if cfg.optimizer == "sgd":
        return SGD(pg, lr=0.1, momentum=0.9, nesterov=True)
    if cfg.optimizer == "adam":
        return Adam(pg, lr=1e-3, betas=(0.9, 0.999))
    raise ValueError(cfg.optimizer)

def build_scheduler(optimizer, cfg, num_epochs: int):
    if cfg.scheduler == "none":
        return None
    if cfg.scheduler == "step":
        main = StepLR(optimizer, step_size=max(1, num_epochs//3), gamma=0.1)
    elif cfg.scheduler == "cosine":
        main = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=0.0)
    else:
        raise ValueError(cfg.scheduler)
    if cfg.warmup:
        warm = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=min(5, max(1, num_epochs//10)))
        return SequentialLR(optimizer, [warm, main], milestones=[warm.total_iters])
    return main

# ========= Mixup & metrics =========
def mixup_batch(x, y, alpha: float = 0.2):
    if alpha <= 0.0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[perm], y, y[perm], lam

def accuracy_from_logits(logits, targets):
    preds = logits.argmax(1)
    return (preds == targets).sum().item(), targets.size(0)

# ========= Train / Eval =========
def train_one_epoch(model, loader, criterion, optimizer, scaler, use_mixup=False, mixup_alpha=0.2, use_amp=False):
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        if use_mixup:
            xb, ya, yb2, lam = mixup_batch(xb, yb, alpha=mixup_alpha)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=device.type, enabled=use_amp):
            out = model(xb)
            loss = (lam * criterion(out, ya) + (1-lam) * criterion(out, yb2)) if use_mixup else criterion(out, yb)
        if scaler.is_enabled():
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); optimizer.step()
        run_loss += loss.item() * xb.size(0)
        c, t = accuracy_from_logits(out, yb)  # proxy under mixup
        correct += c; total += t
    return {"loss": run_loss / max(1,total), "acc": 100.0 * correct / max(1,total)}

@torch.no_grad()
def evaluate(model, loader, criterion, use_amp=False):
    model.eval()
    run_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        with autocast(device_type=device.type, enabled=use_amp):
            if USE_TTA:
                out = _predict_with_tta(model, xb, shifts=TTA_SHIFTS)
                # compute loss on the averaged logits (well-defined)
                loss = criterion(out, yb)
            else:
                out = model(xb)
                loss = criterion(out, yb)
        run_loss += loss.item() * xb.size(0)
        c = (out.argmax(1) == yb).sum().item()
        correct += c; total += yb.size(0)
    return {"loss": run_loss / max(1,total), "acc": 100.0 * correct / max(1,total)}

def evaluate_snr_curve(model, loader_snr):
    model.eval()
    preds, targets, snrs = [], [], []
    with torch.no_grad():
        for xb, yb, sb in loader_snr:
            xb = xb.to(device); yb = yb.to(device)
            if USE_TTA:
                out = _predict_with_tta(model, xb, shifts=TTA_SHIFTS)
            else:
                out = model(xb)
            preds.extend(out.argmax(1).cpu().numpy())
            targets.extend(yb.cpu().numpy())
            snrs.extend(sb.numpy())  # sb is CPU tensor
    preds   = np.array(preds)
    targets = np.array(targets)
    snrs    = np.array(snrs)
    snr_list = sorted(list(set(snrs)))
    accs = []
    for s in snr_list:
        idx = (snrs == s)
        accs.append(100.0 * np.mean(preds[idx] == targets[idx]))
    test_acc = 100.0 * float(np.mean(preds == targets))
    return snr_list, accs, test_acc



# ========= Configs: CONTROL + one-factor ablations =========
@dataclass(frozen=True)
class Cfg:
    name: str
    mixup: bool
    label_smoothing: float
    optimizer: str       # "sgd" | "adam"
    scheduler: str       # "cosine" | "step" | "none"
    warmup: bool
    no_bias_decay: bool  # True => do NOT decay BN/bias
    amp: bool
    seed: int = 42

CONTROL = Cfg("CONTROL", False, 0.1, "sgd", "step", False, True, True, 42)  # StepLR + no warmup baseline

ABLATIONS: List[Cfg] = [
    CONTROL,
    Cfg("A_mixup_on",         True,  CONTROL.label_smoothing, CONTROL.optimizer, CONTROL.scheduler, CONTROL.warmup, CONTROL.no_bias_decay, CONTROL.amp, CONTROL.seed),
    Cfg("A_ls_off",           CONTROL.mixup, 0.0,             CONTROL.optimizer, CONTROL.scheduler, CONTROL.warmup, CONTROL.no_bias_decay, CONTROL.amp, CONTROL.seed),
    Cfg("A_sched_cosine",     CONTROL.mixup, CONTROL.label_smoothing, CONTROL.optimizer, "cosine", CONTROL.warmup, CONTROL.no_bias_decay, CONTROL.amp, CONTROL.seed),
    Cfg("A_warmup_on",        CONTROL.mixup, CONTROL.label_smoothing, CONTROL.optimizer, CONTROL.scheduler, True,  CONTROL.no_bias_decay, CONTROL.amp, CONTROL.seed),
    Cfg("A_decay_all_params", CONTROL.mixup, CONTROL.label_smoothing, CONTROL.optimizer, CONTROL.scheduler, CONTROL.warmup, False, CONTROL.amp, CONTROL.seed),
    Cfg("A_opt_adam",         CONTROL.mixup, CONTROL.label_smoothing, "adam",            CONTROL.scheduler, CONTROL.warmup, CONTROL.no_bias_decay, CONTROL.amp, CONTROL.seed),
    Cfg("A_amp_off",          CONTROL.mixup, CONTROL.label_smoothing, CONTROL.optimizer, CONTROL.scheduler, CONTROL.warmup, CONTROL.no_bias_decay, False, CONTROL.seed),
]

# ========= Single experiment runner =========
def run_experiment(cfg: Cfg, num_epochs: int, batch_size: int = 256, out_dir: str = DRIVE_ROOT, normalize: str = NORM) -> Dict:
    os.makedirs(out_dir, exist_ok=True); os.makedirs(PLOTS_DIR, exist_ok=True); os.makedirs(MODELS_DIR, exist_ok=True)
    set_global_seed(cfg.seed)

    # REBUILD loaders per seed & batch size
    tr_loader, va_loader, te_loader, te_loader_snr = make_loaders_with_snr(
        X_train, y_train, snr_train, X_val, y_val, snr_val, X_test, y_test, snr_test,
        batch_size=batch_size, seed=cfg.seed, normalize=normalize
    )

    model = AMC1DCNN(num_classes).to(device)
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg, num_epochs=num_epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    use_amp = cfg.amp and (device.type == "cuda")
    scaler = GradScaler(enabled=use_amp)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    best_val_acc, best_state = -1.0, None

    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()

    for epoch in range(1, num_epochs+1):
        tr = train_one_epoch(model, tr_loader, criterion, optimizer, scaler,
                             use_mixup=cfg.mixup, mixup_alpha=0.2, use_amp=use_amp)
        va = evaluate(model, va_loader, criterion, use_amp=use_amp)

        history["train_loss"].append(tr["loss"]); history["train_acc"].append(tr["acc"])
        history["val_loss"].append(va["loss"]);   history["val_acc"].append(va["acc"])
        if scheduler is not None: scheduler.step()
        if va["acc"] > best_val_acc:
            best_val_acc = va["acc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"[{cfg.name}] Ep {epoch:03d}/{num_epochs:03d}  "
              f"Tr {tr['loss']:.4f}/{tr['acc']:.2f}%  Va {va['loss']:.4f}/{va['acc']:.2f}%  "
              f"LR {optimizer.param_groups[0]['lr']:.5f}")
        history["lr"].append(float(optimizer.param_groups[0]["lr"]))

    if device.type == "cuda": torch.cuda.synchronize()
    elapsed_sec = time.perf_counter() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    snrs, snr_accs, test_acc = evaluate_snr_curve(model, te_loader_snr)


      # Save model + manifest + history
    ckpt_path = os.path.join(MODELS_DIR, f"1dcnn_{cfg.name}.pth")
    torch.save(model.state_dict(), ckpt_path)

    # right before building `manifest` in run_experiment(...)
    try:
        import subprocess
        repo_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR).decode().strip()
    except Exception:
        repo_commit = "unknown"

    manifest = {
        "name": cfg.name,
        "cfg": asdict(cfg),
        "val_best_acc": best_val_acc,
        "test_acc": test_acc,
        "elapsed_sec": elapsed_sec,
        "num_params": int(sum(p.numel() for p in model.parameters())),
        "ckpt_path": ckpt_path,
        "repo_commit": repo_commit,  # <- add here
    }

    with open(os.path.join(DRIVE_ROOT, f"{cfg.name}_manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)

    np.savez_compressed(os.path.join(out_dir, f"{cfg.name}_history.npz"),
        train_loss=np.array(history["train_loss"], dtype=np.float32),
        train_acc=np.array(history["train_acc"], dtype=np.float32),
        val_loss=np.array(history["val_loss"], dtype=np.float32),
        val_acc=np.array(history["val_acc"], dtype=np.float32),
        lr=np.array(history["lr"], dtype=np.float32)   # <- add
    )

    # Per-run plots
    plt.figure(); plt.plot(history["train_acc"], label="train_acc"); plt.plot(history["val_acc"], label="val_acc")
    plt.xlabel("epoch"); plt.ylabel("accuracy (%)"); plt.title(f"{cfg.name} accuracy"); plt.legend()
    plt.savefig(os.path.join(PLOTS_DIR, f"{cfg.name}_acc.png"), dpi=300, bbox_inches="tight"); plt.close()

    plt.figure(); plt.plot(history["train_loss"], label="train_loss"); plt.plot(history["val_loss"], label="val_loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title(f"{cfg.name} loss"); plt.legend()
    plt.savefig(os.path.join(PLOTS_DIR, f"{cfg.name}_loss.png"), dpi=300, bbox_inches="tight"); plt.close()

    return {
        "name": cfg.name,
        "history": history,
        "snrs": snrs,
        "snr_curve": snr_accs,
        "test_acc": test_acc,
        "val_best_acc": best_val_acc,
        "elapsed_sec": elapsed_sec,
        "ckpt_path": ckpt_path,
    }

# ========= Run all ablations =========
def run_all(num_epochs: int = 30, batch_size: int = 256):
    results = []
    for cfg in ABLATIONS:
        print("\n==============================")
        print(f"Running: {cfg.name}")
        print("==============================")
        res = run_experiment(cfg, num_epochs=num_epochs, batch_size=batch_size)
        results.append(res)

    # Save table to Drive
    import pandas as pd
    df = pd.DataFrame([{
        "Experiment": r["name"],
        "Val Best Acc (%)": r["val_best_acc"],
        "Test Acc (%)": r["test_acc"],
        "Train Time (s)": r["elapsed_sec"]
    } for r in results])
    df_path_csv = os.path.join(PLOTS_DIR, "ablation_results.csv")
    df_path_pkl = os.path.join(PLOTS_DIR, "ablation_results.pkl")
    df.to_csv(df_path_csv, index=False); df.to_pickle(df_path_pkl)
    print("Saved results:", df_path_csv, "and", df_path_pkl)

    # Plot val acc curves for all configs
    plt.figure(figsize=(10,5))
    for r in results:
        plt.plot(r["history"]["val_acc"], label=r["name"])
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy (%)")
    plt.title("Validation Accuracy vs Epoch (All Configurations)")
    plt.legend()
    plt.savefig(os.path.join(PLOTS_DIR, "val_accuracy_all_configs.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # Plot loss curves (train/val) for all configs
    plt.figure(figsize=(10,5))
    for r in results:
        plt.plot(r["history"]["train_loss"], "--", alpha=0.7, label=f"{r['name']} Train")
        plt.plot(r["history"]["val_loss"], alpha=0.9, label=f"{r['name']} Val")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.title("Training & Validation Loss vs Epoch (All Configurations)")
    plt.legend(ncol=2)
    plt.savefig(os.path.join(PLOTS_DIR, "loss_curves_all_configs.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # SNR accuracy comparison for all configs
    plt.figure(figsize=(12,6))
    for r in results:
        plt.plot(r["snrs"], r["snr_curve"], marker='o', linewidth=2, label=r["name"])
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.xlabel("SNR (dB)"); plt.ylabel("Accuracy(%)")
    plt.title("Ablation Study: Accuracy vs SNR")
    plt.legend()
    plt.savefig(os.path.join(PLOTS_DIR, "snr_accuracy_comparison.png"), dpi=300, bbox_inches='tight')
    plt.close()

    # BEST model learning curves
    best = max(results, key=lambda r: r["val_best_acc"])
    best_name = best["name"]; best_hist = best["history"]
    plt.figure(); plt.plot(best_hist["train_acc"], label="train_acc"); plt.plot(best_hist["val_acc"], label="val_acc")
    plt.xlabel("epoch"); plt.ylabel("accuracy (%)"); plt.title(f"BEST ({best_name}) accuracy"); plt.legend()
    plt.savefig(os.path.join(PLOTS_DIR, "BEST_acc.png"), dpi=300, bbox_inches="tight"); plt.close()

    plt.figure(); plt.plot(best_hist["train_loss"], label="train_loss"); plt.plot(best_hist["val_loss"], label="val_loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title(f"BEST ({best_name}) loss"); plt.legend()
    plt.savefig(os.path.join(PLOTS_DIR, "BEST_loss.png"), dpi=300, bbox_inches="tight"); plt.close()

    # ---------- NEW: bundle best model in its own folder ----------
    best_dir = os.path.join(DRIVE_ROOT, "best_model")
    os.makedirs(best_dir, exist_ok=True)

    # paths already produced earlier in the script
    best_ckpt_path     = best["ckpt_path"]  # e.g. .../models/1dcnn_CONTROL.pth
    best_manifest_path = os.path.join(DRIVE_ROOT, f"{best_name}_manifest.json")
    best_history_path  = os.path.join(DRIVE_ROOT, f"{best_name}_history.npz")
    best_acc_plot      = os.path.join(PLOTS_DIR, "BEST_acc.png")
    best_loss_plot     = os.path.join(PLOTS_DIR, "BEST_loss.png")

    for src, dst_name in [
        (best_ckpt_path,     "best_model.pth"),
        (best_manifest_path, "best_manifest.json"),
        (best_history_path,  "best_history.npz"),
        (best_acc_plot,      "best_acc.png"),
        (best_loss_plot,     "best_loss.png"),
    ]:
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(best_dir, dst_name))

    # also save the SNR curve + test acc as JSON for quick comparison later
    with open(os.path.join(best_dir, "best_snr_curve.json"), "w") as f:
        json.dump(
            {
                "name": best_name,
                "snrs": [float(s) for s in best["snrs"]],
                "accuracy": [float(a) for a in best["snr_curve"]],
                "test_acc": float(best["test_acc"]),
                "val_best_acc": float(best["val_best_acc"]),
            },
            f,
            indent=2,
        )
    print("Best model bundle saved under:", best_dir)
    # -------------------------------------------------------------

    # Save all histories as a single pickle
    histories = {r["name"]: r["history"] for r in results}
    with open(os.path.join(PLOTS_DIR, "histories.pkl"), "wb") as f:
        pickle.dump(histories, f)

    print("\n=== DONE ===")
    print("Artifacts in:", DRIVE_ROOT)
    return results



# ========= RUN! =========
set_global_seed(42)
if RUN_SINGLE:
    results = run_all(num_epochs=35, batch_size=256)

    auc_rows = []
    for r in results:
        auc = compute_auc_over_snr(r["snrs"], r["snr_curve"], normalize=True)
        auc_rows.append((r["name"], auc))
    auc_rows.sort(key=lambda x: x[1], reverse=True)

if RUN_SINGLE:
    plt.figure(figsize=(10,5))
    plt.bar([k for k,_ in auc_rows], [v for _,v in auc_rows])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("AUC-over-SNR (mean accuracy %)")
    plt.title("AUC-over-SNR by Configuration")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "auc_over_snr_bar.png"), dpi=300, bbox_inches="tight"); plt.close()

    # LR curves for all configs
    plot_lr_curves_for_all(results, PLOTS_DIR)

    # Confusion matrix & reliability for BEST model (by val_best_acc)
    best = max(results, key=lambda r: r["val_best_acc"])
    best_snr_list = best["snrs"]
    target_snr = max(best_snr_list) if len(best_snr_list) else 18.0
    plot_confusion_at_snr(best["ckpt_path"], best["name"], target_snr, label_names, PLOTS_DIR)
    plot_reliability_and_ece(best["ckpt_path"], best["name"], test_loader, PLOTS_DIR)

Python: 3.12.12  |  Platform: Linux-6.6.105+-x86_64-with-glibc2.35
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Artifacts will go to: /content/drive/MyDrive/amc_runs/cnn_1d/20251130-044350
Using device: cuda
Loading RadioML 2016.10A from: /content/drive/MyDrive/AMC_datasets/RadioML2016_10A_dataset.pkl
Loading RadioML 2016.10A dataset from: /content/drive/MyDrive/AMC_datasets/RadioML2016_10A_dataset.pkl
Found 11 raw modulation types.
Loaded dataset contains 11 modulation classes and 20 SNR values.
Dataset summary: 11 mods, 20 SNRs, 220,000 total samples.
X shape: (220000, 2, 128) | y shape: (220000,) | snr shape: (220000,)
Unique mods: [np.str_('8PSK'), np.str_('AM-DSB'), np.str_('AM-SSB'), np.str_('BPSK'), np.str_('CPFSK'), np.str_('GFSK'), np.str_('PAM4'), np.str_('QAM16'), np.str_('QAM64'), np.str_('QPSK'), np.str_('WBFM')] | Unique SNRs: [np.int64(-20), np.int64(-18), np.int64(-16), np.int64(-14), np

/tmp/ipython-input-1689270060.py:275: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  area = np.trapz(a, s)  # percent * dB


Saved: /content/drive/MyDrive/amc_runs/cnn_1d/20251130-044350/plots/lr_curves_all_configs.png
Saved: /content/drive/MyDrive/amc_runs/cnn_1d/20251130-044350/plots/A_ls_off_confusion_18dB.png
Saved: /content/drive/MyDrive/amc_runs/cnn_1d/20251130-044350/plots/A_ls_off_reliability.png


In [3]:
# === Finalize the best configuration: retrain on Train∪Val for best-epoch length ===
def make_trainval_loader(
    X_tr, y_tr, snr_tr, X_va, y_va, snr_va, batch_size=256, seed=42, normalize=NORM
):
    g = torch.Generator(device="cpu").manual_seed(seed)
    pin = (device.type == "cuda")
    norm_fn = _get_norm_fn(normalize)
    X_tv = np.concatenate([X_tr, X_va], axis=0)
    y_tv = np.concatenate([y_tr, y_va], axis=0)
    Xt = torch.from_numpy(norm_fn(X_tv)).float()
    yt = torch.from_numpy(y_tv).long()
    train_loader_tv = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True, generator=g, pin_memory=pin)
    return train_loader_tv

@torch.no_grad()
def test_metrics_dict(model, test_loader, test_loader_snr):
    ce = nn.CrossEntropyLoss()
    te = evaluate(model, test_loader, ce, use_amp=(device.type=="cuda"))
    snrs, snr_accs, test_acc = evaluate_snr_curve(model, test_loader_snr)
    return {
        "test_acc": float(test_acc),
        "snrs": [float(s) for s in snrs],
        "snr_curve": [float(a) for a in snr_accs],
    }

def retrain_final_from_best(
    results, X_tr, y_tr, snr_tr, X_va, y_va, snr_va,
    num_seeds=(42,43,44), batch_size=256, normalize="none", out_dir=DRIVE_ROOT
):
    os.makedirs(MODELS_DIR, exist_ok=True)
    best = max(results, key=lambda r: r["val_best_acc"])
    best_name = best["name"]
    # epoch length = epoch of best val acc during ablation
    best_epoch = int(np.argmax(np.array(best["history"]["val_acc"])) + 1)

    finals = []
    for seed in num_seeds:
        set_global_seed(seed)
        train_loader_tv = make_trainval_loader(
            X_tr, y_tr, snr_tr, X_va, y_va, snr_va, batch_size=batch_size, seed=seed, normalize=normalize
        )

        # Build model/optim/sched from the winning config
        cfg = next(c for c in ABLATIONS if c.name == best_name)
        model = AMC1DCNN(num_classes).to(device)
        optimizer = build_optimizer(model, cfg)
        scheduler = build_scheduler(optimizer, cfg, num_epochs=best_epoch)
        criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
        use_amp = cfg.amp and (device.type == "cuda")
        scaler = GradScaler(enabled=use_amp)

        # Train exactly for 'best_epoch' epochs on Train∪Val
        for ep in range(1, best_epoch + 1):
            _ = train_one_epoch(model, train_loader_tv, criterion, optimizer, scaler,
                                use_mixup=cfg.mixup, mixup_alpha=0.2, use_amp=use_amp)
            if scheduler is not None:
                scheduler.step()

        # Evaluate on the held-out Test only
        metrics = test_metrics_dict(model, test_loader, test_loader_snr)

        # Save final checkpoint + manifest
        fname = f"FINAL_{best_name}_seed{seed}.pth"
        fpath = os.path.join(MODELS_DIR, fname)
        torch.save(model.state_dict(), fpath)
        try:
            repo_commit = subprocess.check_output(
                ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], stderr=subprocess.DEVNULL
            ).decode().strip()
        except Exception:
            repo_commit = None
        manifest = {
            "name": cfg.name,
            "cfg": asdict(cfg),
            "best_epoch_used": int(best_epoch),
            "test_acc": float(metrics["test_acc"]),            # %
            "snrs": metrics["snrs"],
            "snr_curve": metrics["snr_curve"],                 # %
            "num_params": int(sum(p.numel() for p in model.parameters())),
            "ckpt_path": fpath,
            "normalize": NORMALIZE_MODE,
            "repo_commit": repo_commit,
        }
        with open(os.path.join(out_dir, f"FINAL_{best_name}_seed{seed}.json"), "w") as f:
            json.dump(manifest, f, indent=2)

        finals.append(manifest)
        print(f"[FINAL] {best_name} seed={seed}  test_acc={metrics['test_acc']:.2f}%  saved={fpath}")

    # Aggregate across seeds
    accs = [m["test_acc"] for m in finals]
    mean_acc = float(np.mean(accs)); std_acc = float(np.std(accs, ddof=1)) if len(accs) > 1 else 0.0
    print(f"[FINAL] mean±std test acc over {len(accs)} seeds: {mean_acc:.2f}% ± {std_acc:.2f}%")
    with open(os.path.join(out_dir, f"FINAL_{best_name}_summary.json"), "w") as f:
        json.dump({"mean_test_acc": mean_acc, "std_test_acc": std_acc, "runs": finals}, f, indent=2)
    return finals


if RUN_FINAL_OF_BEST:
    final_runs = retrain_final_from_best(
    results, X_train, y_train, snr_train, X_val, y_val, snr_val,
    num_seeds=(42,43,44), batch_size=256, normalize=NORM, out_dir=DRIVE_ROOT
)


[FINAL] A_ls_off seed=42  test_acc=61.74%  saved=/content/drive/MyDrive/amc_runs/cnn_1d/20251130-044350/models/FINAL_A_ls_off_seed42.pth
[FINAL] A_ls_off seed=43  test_acc=61.83%  saved=/content/drive/MyDrive/amc_runs/cnn_1d/20251130-044350/models/FINAL_A_ls_off_seed43.pth
[FINAL] A_ls_off seed=44  test_acc=61.67%  saved=/content/drive/MyDrive/amc_runs/cnn_1d/20251130-044350/models/FINAL_A_ls_off_seed44.pth
[FINAL] mean±std test acc over 3 seeds: 61.75% ± 0.08%


In [4]:
def _acc_of(name: str, results):
    for r in results:
        if r["name"] == name:
            return r["val_best_acc"]
    return None

def build_ultimate_from_results(results, control_cfg=CONTROL):
    control_acc = _acc_of(control_cfg.name, results)
    if control_acc is None:
        raise RuntimeError("CONTROL result not found")

    # Decide each knob by simple win-vs-control rule
    mixup = (_acc_of("A_mixup_on", results) or -1) > control_acc
    ls     = 0.0 if (_acc_of("A_ls_off", results) or -1) > control_acc else control_cfg.label_smoothing
    sched  = "cosine" if (_acc_of("A_sched_cosine", results) or -1) > control_acc else control_cfg.scheduler
    warmup = True if (_acc_of("A_warmup_on", results) or -1) > control_acc else control_cfg.warmup
    # no_bias_decay=True means "do NOT decay BN/bias"; if decaying all params was better, flip it to False
    no_bias_decay = False if (_acc_of("A_decay_all_params", results) or -1) > control_acc else control_cfg.no_bias_decay
    optimizer = "adam" if (_acc_of("A_opt_adam", results) or -1) > control_acc else control_cfg.optimizer
    amp = control_cfg.amp if (_acc_of("A_amp_off", results) or -1) <= control_acc else False  # keep AMP unless 'amp_off' wins

    return Cfg(
        name="ULTIMATE_auto",
        mixup=mixup,
        label_smoothing=ls,
        optimizer=optimizer,
        scheduler=sched,
        warmup=warmup,
        no_bias_decay=no_bias_decay,
        amp=amp,
        seed=42,
    )

# 1) Build combined config from your ablation outcomes
if RUN_ULTIMATE:
    ULTIMATE = build_ultimate_from_results(results)
    print("\n==============================")
    print(f"Running: {ULTIMATE.name} (combined)")
    print("==============================")
    ultimate_res = run_experiment(ULTIMATE, num_epochs=100, batch_size=BATCH_SIZE, normalize=NORM)

    best_E = best_epoch_from_history(ultimate_res["history"])

    # train on Train∪Val for best_E epochs, then final test (your code here unchanged)

    # 3) (Optional) Retrain final on Train∪Val for the best-epoch length discovered for ULTIMATE


    # change this line (currently hard-coded to "none"):
    train_loader_tv = (lambda bs=256, seed=42, norm=NORM:
        DataLoader(
            TensorDataset(
                torch.from_numpy(_get_norm_fn(norm)(np.concatenate([X_train, X_val],0))).float(),
                torch.from_numpy(np.concatenate([y_train, y_val],0)).long(),
            ),
            batch_size=bs, shuffle=True, generator=torch.Generator().manual_seed(seed),
            pin_memory=(device.type=="cuda"))
    )()

    model = AMC1DCNN(num_classes).to(device)
    opt = build_optimizer(model, ULTIMATE)
    sch = build_scheduler(opt, ULTIMATE, num_epochs=best_E)
    crit = nn.CrossEntropyLoss(label_smoothing=ULTIMATE.label_smoothing)
    use_amp = ULTIMATE.amp and (device.type=="cuda")
    scaler = GradScaler(enabled=use_amp)
    for ep in range(1, best_E+1):
        _ = train_one_epoch(model, train_loader_tv, crit, opt, scaler,
                            use_mixup=ULTIMATE.mixup, mixup_alpha=0.2, use_amp=use_amp)
        if sch is not None: sch.step()

    # Final test on the held-out test set
    final_test = evaluate(model, test_loader, crit, use_amp=use_amp)
    snrs, snr_curve, test_acc = evaluate_snr_curve(model, test_loader_snr)
    print(f"[FINAL-ULTIMATE] test_acc={test_acc:.2f}%")


In [5]:
# ========= Multi-seed ablation harness =========

from collections import defaultdict
import pandas as pd

def _with_seed(cfg: Cfg, seed: int, name_suffix: str):
    return Cfg(
        name=f"{cfg.name}_seed{seed}{name_suffix}",
        mixup=cfg.mixup,
        label_smoothing=cfg.label_smoothing,
        optimizer=cfg.optimizer,
        scheduler=cfg.scheduler,
        warmup=cfg.warmup,
        no_bias_decay=cfg.no_bias_decay,
        amp=cfg.amp,
        seed=seed,
    )

def _base_name(run_name: str) -> str:
    # strips trailing _seedXX or similar; keeps original ablation name
    if "_seed" in run_name:
        return run_name.split("_seed")[0]
    return run_name

def run_all_multi_seed(num_epochs: int, batch_size: int, seeds=(42,43,44)):
    """
    Runs every ablation in ABLATIONS across multiple seeds.
    Returns a list of per-seed run dicts (same schema as run_experiment),
    plus writes raw & summary CSVs and an errorbar plot to Drive.
    """
    per_seed_runs = []
    print("\n=== MULTI-SEED ABLATIONS ===")
    for cfg in ABLATIONS:
        for s in seeds:
            print("\n------------------------------")
            print(f"Running: {cfg.name}  (seed={s})")
            print("------------------------------")
            cfg_s = _with_seed(cfg, s, name_suffix="")  # name includes seed
            res = run_experiment(cfg_s, num_epochs=num_epochs, batch_size=batch_size)
            per_seed_runs.append(res)

    # ---------- Raw table ----------
    rows = []
    for r in per_seed_runs:
        rows.append({
            "Experiment": _base_name(r["name"]),
            "RunName": r["name"],
            "Seed": int(r["name"].split("seed")[-1]) if "seed" in r["name"] else None,
            "Val Best Acc (%)": r["val_best_acc"],
            "Test Acc (%)": r["test_acc"],
            "Train Time (s)": r["elapsed_sec"],
            "AUC over SNR (mean acc %)": compute_auc_over_snr(r["snrs"], r["snr_curve"], normalize=True)
        })
    df_raw = pd.DataFrame(rows)
    raw_csv = os.path.join(PLOTS_DIR, "ablation_multi_seed_raw.csv")
    df_raw.to_csv(raw_csv, index=False)

    # ---------- Summary (mean±std) ----------
    g = df_raw.groupby("Experiment")
    df_sum = pd.DataFrame({
        "Experiment": g.size().index,
        "Seeds": g.size().values,
        "Val Best Acc mean (%)": g["Val Best Acc (%)"].mean().values,
        "Val Best Acc std (%)": g["Val Best Acc (%)"].std(ddof=1).fillna(0).values,
        "Test Acc mean (%)": g["Test Acc (%)"].mean().values,
        "Test Acc std (%)": g["Test Acc (%)"].std(ddof=1).fillna(0).values,
        "AUC mean (%)": g["AUC over SNR (mean acc %)"].mean().values,
        "AUC std (%)": g["AUC over SNR (mean acc %)"].std(ddof=1).fillna(0).values,
        "Avg Train Time (s)": g["Train Time (s)"].mean().values
    }).sort_values("Test Acc mean (%)", ascending=False)
    sum_csv = os.path.join(PLOTS_DIR, "ablation_multi_seed_summary.csv")
    df_sum.to_csv(sum_csv, index=False)

    # ---------- Error-bar bar chart (mean±std test acc) ----------
    plt.figure(figsize=(10,5))
    x = np.arange(len(df_sum))
    means = df_sum["Test Acc mean (%)"].values
    stds  = df_sum["Test Acc std (%)"].values
    plt.bar(df_sum["Experiment"].values, means, yerr=stds, capsize=4)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Test Accuracy (%)")
    plt.title("Multi-seed Ablations: mean±std Test Acc")
    plt.tight_layout()
    err_png = os.path.join(PLOTS_DIR, "multi_seed_testacc_errorbar.png")
    plt.savefig(err_png, dpi=300, bbox_inches="tight"); plt.close()

    # ---------- Pick the winner ----------
    winner_row = df_sum.iloc[0]
    winner_name = str(winner_row["Experiment"])
    print("\n=== MULTI-SEED SUMMARY (sorted by Test Acc mean) ===")
    print(df_sum.to_string(index=False))
    print(f"\n>>> Winner by Test Acc mean: {winner_name} "
          f"(mean={winner_row['Test Acc mean (%)']:.2f}%, std={winner_row['Test Acc std (%)']:.2f}%)")
    print("Saved:", raw_csv, "and", sum_csv, "and", err_png)

    return per_seed_runs, df_raw, df_sum, winner_name


def _cfg_lookup_by_name(name: str) -> Cfg:
    for c in ABLATIONS:
        if c.name == name:
            return c
    raise KeyError(f"Config {name} not in ABLATIONS")

def _make_trainval_loader(batch_size=256, seed=42, normalize=NORM):
    # uses global X_train, X_val, y_train, y_val from earlier in the notebook
    g = torch.Generator(device="cpu").manual_seed(seed)
    pin = (device.type == "cuda")
    X_tv = np.concatenate([X_train, X_val], 0)
    y_tv = np.concatenate([y_train, y_val], 0)
    X_tv = _get_norm_fn(normalize)(X_tv).astype(np.float32)
    Xt = torch.from_numpy(X_tv).float()
    yt = torch.from_numpy(y_tv).long()
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=True, generator=g, pin_memory=pin)

def final_retrain_for_winner(winner_name: str, per_seed_runs, seeds=(42,43,44),
                             normalize="none", save_prefix="FINAL"):
    base_cfg = _cfg_lookup_by_name(winner_name)
    finals = []
    for s in seeds:
        # Find the matching exploratory run for this seed
        matches = [r for r in per_seed_runs if (_base_name(r["name"])==winner_name and f"seed{s}" in r["name"])]
        best_E = best_epoch_from_history(matches[0]["history"]) if matches else 100


        # build model and retrain on Train∪Val
        set_global_seed(s)
        loader_tv = _make_trainval_loader(batch_size=256, seed=s, normalize=normalize)

        model = AMC1DCNN(num_classes).to(device)
        opt = build_optimizer(model, base_cfg)
        sch = build_scheduler(opt, base_cfg, num_epochs=best_E)
        crit = nn.CrossEntropyLoss(label_smoothing=base_cfg.label_smoothing)
        use_amp = base_cfg.amp and (device.type == "cuda")
        scaler = GradScaler(enabled=use_amp)

        for ep in range(1, best_E+1):
            train_one_epoch(model, loader_tv, crit, opt, scaler,
                            use_mixup=base_cfg.mixup, mixup_alpha=0.2, use_amp=use_amp)
            if sch is not None: sch.step()

        # final test
        te = evaluate(model, test_loader, crit, use_amp=use_amp)
        snrs, snr_curve, test_acc = evaluate_snr_curve(model, test_loader_snr)
        finals.append(test_acc)

        # save
        ck = os.path.join(MODELS_DIR, f"{save_prefix}_{winner_name}_seed{s}.pth")
        torch.save(model.state_dict(), ck)
        print(f"[FINAL] {winner_name} seed={s}  test_acc={test_acc:.2f}%  saved={ck}")

    finals = np.array(finals, dtype=float)
    print(f"[FINAL] mean±std test acc over {len(seeds)} seeds: {finals.mean():.2f}% ± {finals.std(ddof=1):.2f}%")
    return finals

# ====== RUN IT ======
SEEDS = (42, 43, 44)
if RUN_MULTI:
    per_seed_runs, df_raw, df_sum, winner_name = run_all_multi_seed(num_epochs=100, batch_size=256, seeds=(42,43,44))
    finals = final_retrain_for_winner(winner_name, per_seed_runs, seeds=(42,43,44), normalize=NORM, save_prefix="FINAL")

if RUN_FINAL_OF_BEST:
    best = max(results, key=lambda r: r["val_best_acc"])
    paths = [os.path.join(MODELS_DIR, f"FINAL_{best['name']}_seed{s}.pth") for s in (42,43,44)]
    ens_acc = eval_seed_ensemble(paths, test_loader, num_classes)
    print(f"[FINAL-ENSEMBLE] seeds 42/43/44  test_acc={ens_acc:.2f}%")
    with open(os.path.join(DRIVE_ROOT, f"FINAL_{best['name']}_ensemble.json"), "w") as f:
        json.dump({"seeds":[42,43,44], "test_acc": float(ens_acc), "normalize": NORM}, f, indent=2)


[FINAL-ENSEMBLE] seeds 42/43/44  test_acc=62.19%
